--------------------------------------------------
# **Create Czech text dataset for training**
-------------------------------------------------

In [1]:
!apt install git-lfs




git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 138 not upgraded.


## ***Imports***

In [2]:
import os, sys
import zipfile
import subprocess
import io
import shutil
import json
import polars as pl
import numpy as np
import pyarrow as pa
import pickle

import pyarrow.parquet as pq
import pyarrow.dataset as ds

from huggingface_hub import (
    get_full_repo_name,
    login,
    upload_folder,
    hf_hub_download,
    HfApi
)


from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from tqdm import tqdm

## ***Hugging Face Login***

In [3]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
user_email = user_secrets.get_secret("user_email")
user_name = user_secrets.get_secret("user_name")

login(token=hf_token)

### ***Data files***

1. ***CZLC/cermat_czech_mc***
   
   The **Cermat Czech MultiChoice** dataset was collected from assignments official CERMAT website. The dataset was collected from three tiers of assignments: 6 year, 9 year primary school test and final high school tests (so-called maturita). The assignments were semi-manually extracted from official PDFs available at [CERMAT's website](https://maturita.cermat.cz/menu/testy-a-zadani-z-predchozich-obdobi/matematika/testy-a-zadani-matematika).

    **Collection Date Range:** years 2016-2023

    ***Licensing and Credits***

    The majority of collection work was done by our student co-worker Jan Kapsa. Members of CZLC do not own the test assignments, neither are responsible for their contents.

    **Source:** https://huggingface.co/datasets/CZLC/cermat_czech_mc
 
2. ***hynky/czech-justice-summ-alpaca-long***

    **Source:** https://huggingface.co/datasets/hynky/czech-justice-summ-alpaca-long

3. ***hynky/czech_news_dataset_v2***

    Dataset containing the news articles from major online news outlets collected from 2000-2022.
    Follow-up paper https://arxiv.org/abs/2307.10666 (v1 of the dataset)

    Changes from v1:
        - Better contribution of novinky.cz in later stages
        - More articles, as a mistake in filtering was fixed.

   Collection was done using CmonCrawl.

   The dataset should be used for Research only purposes as I don't have rights for articles itself.

   **Source:** https://huggingface.co/datasets/hynky/czech_news_dataset_v2

4. ***davidadamczyk/czechbench_agree***

   This is an adapted and filtered test subset from the original Czech grammar agreement dataset

   **Source:** https://huggingface.co/datasets/davidadamczyk/czechbench_agree

6. ***davidadamczyk/czechbench_czech_news***

   This is a simplified and subsampled test subset from the original [czech_news_dataset_v2](https://huggingface.co/datasets/hynky/czech_news_dataset_v2).

   **Source:** https://huggingface.co/datasets/davidadamczyk/czechbench_czech_news

8. ***CIIRC-NLP/czech_news_simple-cs***

   **Source:** https://huggingface.co/datasets/CIIRC-NLP/czech_news_simple-cs

   This is a simplified and subsampled test subset from the original czech_news_dataset_v2.

   Only 5 basic news categories are considered:

   - Foreign
   - Local
   - Sports
   - Economy

   The test set includes 200 examples per category, 1000 examples in total. Apart from the category label, each example also contains the article's headline, brief summary, full textual content, optional keywords, original category specification, and URL.

   This dataset was created for use within the [Czech-Bench](https://gitlab.com/jirkoada/czech-bench) evaluation framework. 

In [4]:
ds01 = load_dataset("CZLC/cermat_czech_mc")
ds02 = load_dataset("hynky/czech-justice-summ-alpaca-long")
ds03 = load_dataset("hynky/czech_news_dataset_v2")
ds04 = load_dataset("davidadamczyk/czechbench_agree")
ds05 = load_dataset("davidadamczyk/czechbench_czech_news")
ds06 = load_dataset("CIIRC-NLP/czech_news_simple-cs", "default")

README.md: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/649 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/503 [00:00<?, ?B/s]

data/train-00000-of-00001-93f855e5cee8b5(…):   0%|          | 0.00/12.6M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/20 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/980 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001-8fd5c2a953da2a2(…):   0%|          | 0.00/2.67M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
print(type(ds01),'\n',ds01)
print(type(ds02),'\n',ds02)
print(type(ds03),'\n',ds03)
print(type(ds04),'\n',ds04)
print(type(ds05),'\n',ds05)
print(type(ds06),'\n',ds06)

<class 'datasets.dataset_dict.DatasetDict'> 
 DatasetDict({
    test: Dataset({
        features: ['type', 'query', 'choices', 'gold', 'context', 'subject', 'difficulty', 'source'],
        num_rows: 649
    })
})
<class 'datasets.dataset_dict.DatasetDict'> 
 DatasetDict({
    train: Dataset({
        features: ['output', 'instruction'],
        num_rows: 4560
    })
})
<class 'datasets.dataset_dict.DatasetDict'> 
 DatasetDict({
    train: Dataset({
        features: ['url', 'authors', 'headline', 'brief', 'keywords', 'category', 'content', 'comments_num', 'server', 'category_unclean', 'authors_gender', 'authors_cum_gender', 'day_of_week', 'date'],
        num_rows: 1641471
    })
    validation: Dataset({
        features: ['url', 'authors', 'headline', 'brief', 'keywords', 'category', 'content', 'comments_num', 'server', 'category_unclean', 'authors_gender', 'authors_cum_gender', 'day_of_week', 'date'],
        num_rows: 144836
    })
    test: Dataset({
        features: ['url', 'au

### ***We display data files in polars***

In [6]:
df01 = pl.from_arrow(ds01["test"].data.table)
df01.sample(5)

type,query,choices,gold,context,subject,difficulty,source
str,str,list[str],i64,str,str,str,str
"""MC""","""Ve kterém z následujících úsek…","[""A. ty se zadem dostaneš domů"", ""B. na klacek místo svýho praporu"", … ""D. tak to sem vyřízenej""]",0,"""Když se Prcek vrátil ze zajetí…","""český jazyk a literatura""","""šestiletá gymnázia""","""3. řádný termín 2022"""
"""MC""","""Která z následujících vět není…","[""A. Nejdůležitější kontakty mi v telefonním seznamu bohužel chyběly."", ""B. Dlouho plánovaný varhanní koncert se skutečně vydařil."", … ""D. Tyto dvě zcela odlišné role prokázaly její všestranné nadání.""]",2,null,"""český jazyk a literatura""","""šestiletá gymnázia""","""1. řádný termín 2022"""
"""MC""","""Ve kterém z následujících úsek…","[""A. krutá zkáza ale nepřítele přišla draho"", ""B. zastrč tedy šíp zpět do toulce"", … ""D. pocítíš rány mého kyje""]",3,"""Robin Hood kráčel po stezce Sh…","""český jazyk a literatura""","""šestiletá gymnázia""","""2. řádný termín 2022"""
"""MC""","""Které z následujících tvrzení …","[""A. Chybně užité slovo laik by mělo být nahrazeno slovem expert a chybně užité slovo minorita by mělo být nahrazeno slovem majorita."", ""B. Význam slov laik a minorita odpovídá danému kontextu, žádné z nich tedy není nutné nahradit jiným slovem."", … ""D. Význam slova laik odpovídá danému kontextu, chybně užité slovo minorita by však mělo být nahrazeno slovem majorita.""]",1,"""<bold>(1)</bold> Vexilologie s…","""český jazyk a literatura""","""maturitní zkouška""","""podzim 2022"""
"""MC""","""Která z následujících informac…","[""A. informace, že s nápadem napsat zprávu na oholenou hlavu otroka přišli staří Řekové"", ""B. informace, že šifrované zprávy se používaly už za dob Julia Caesara"", … ""D. informace, že na dnešní poměry je Caesarova šifra snadno rozluštitelná""]",0,"""Zvláštní způsob, jak skrýt obs…","""český jazyk a literatura""","""maturitní zkouška""","""jaro 2019"""


In [7]:
df02 = pl.from_arrow(ds03["train"].data.table)
df02.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://www.idnes.cz/onadnes/v…","[""Bibiana Martina Hykl""]","""Plus size modelka chce být jak…","""Plus size modelka Eudoxie Yao …","[""Austrálie"", ""Pobřeží slonoviny"", … ""Móda""]",21,"""Eudoxie žije na Pobřeží slonov…",45,2,"""Ona""",[2],2,4,2016-11-03 00:00:00
"""https://www.idnes.cz/sport/mot…",[],"""Splní dominantní mercedesy rol…","""Vozy s velkou maximální rychlo…","[""Ferrari"", ""Mercedes-benz"", … ""Mezinárodní plavecká federace (fina)""]",0,"""Mercedes vyhrál všechny závody…",null,2,null,[],0,4,2014-08-21 00:00:00
"""https://valassky.denik.cz/neho…","[""Dušan Póč""]","""Náraz auta do vlaku se obešel …","""Halenkov – Ohromné štěstí stál…","[""Automobil"", ""Vlak"", … ""Dopravní nehody""]",0,"""Ke střetu vlaku jedoucího z Ve…",null,5,null,[1],1,1,2016-01-11 00:00:00
"""https://www.novinky.cz/zahrani…",[],"""Írán přesouvá centrifugy na ob…","""Írán začal stěhovat centrifugy…",[],1,"""„Přesun centrifug z Natanzu do…",null,4,"""Zahraniční""",[],0,1,2011-08-22 00:00:00
"""https://zpravy.aktualne.cz/dom…",[],"""Vládní krize končí, rozhádaná …","""ODS, TOP 09 a VV se v úterý do…","[""Ekonomický ministr"", ""Vv"", … ""Tomáš jarolím""]",2,"""Praha - Týden poté, co Věci ve…",null,3,"""Politika""",[],0,3,2012-04-11 00:00:00


In [8]:
df03_train = pl.from_arrow(ds03["train"].data.table)
df03_train.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://www.novinky.cz/koktejl…",[],"""Twerkující babičky se staly se…","""Basketbalový celek Washington …","[""Usa"", ""Washington wizards"", … ""Koktejl""]",6,"""V týmu Wizdom Dancers je 19 že…",null,4,"""Koktejl""",[],0,7,2019-04-07 00:00:00
"""https://www.novinky.cz/ekonomi…",[],"""Světové automobilky nevyděláva…","""Několik tisíc lidí hodlá propu…","[""Rolls-royce"", ""Peugeot"", … ""Ekonomika""]",7,"""Francouzská automobilka bude p…",null,4,"""Ekonomika""",[],0,3,2018-01-24 00:00:00
"""https://www.novinky.cz/koktejl…",[],"""Opilý zloděj uvízl v šachtě, s…","""Policisté a hasiči museli ne c…",[],6,"""„Zpočátku se snažil dostat do …",null,4,"""Koktejl""",[],0,3,2019-04-03 00:00:00
"""https://www.novinky.cz/kultura…","[""Karel Černý""]","""Nad knihou: Další zpráva o ban…","""„V Sidi Moumen patřila smrt ke…",[],4,"""Navečer 16. května 2003 se v C…",null,4,"""Kultura""",[1],1,4,2015-04-09 00:00:00
"""https://www.novinky.cz/ekonomi…",[],"""Nabídky operátorů chápe minimu…","""V nabídkách mobilních operátor…","[""Operátoři"", ""Tarify"", … ""Ekonomika""]",7,"""„Řešením je jasná, otevřená a …",null,4,"""Ekonomika""",[],0,3,2012-12-12 00:00:00


In [9]:
df03_validation = pl.from_arrow(ds03["validation"].data.table)
df03_validation.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://kolinsky.denik.cz/z-re…","[""Milan Holakovský""]","""Zeleň za miliony má mazat uhlí…","""Výsadba zeleně na 651 místech …","[""Jaroslava jermanová""]",0,"""Byť sázet se má až na jaře, te…",null,5,"""ZPRÁVY""",[1],1,2,2020-06-30 00:00:00
"""https://www.denik.cz/regiony/v…","[""Iva Kovářová""]","""Vězni pomáhají policistům i lé…","""Ochranné látkové roušky šije t…","[""Nebezpečný koronavirus"", ""Výroba"", … ""Vězeňská služba české republiky""]",0,"""On-line reportáž ke koronaviru…",null,5,"""ZPRÁVY""",[2],2,2,2020-04-07 00:00:00
"""https://fm.denik.cz/ostatni_re…","[""Roman Brhel""]","""Stromský bude na dostizích pop…","""Letošní dostihová sezona bude …","[""Dostih"", ""Marek stromský"", … ""Štěpánkovice""]",3,"""„Je dobře, že se začíná. Dle m…",null,5,"""SPORT""",[1],1,3,2020-05-06 00:00:00
"""https://hranicky.denik.cz/zpra…","[""Liba Mátlová""]","""Vánoční strom v Hranicích se r…","""Na první adventní neděli se ro…","[""Vánoční strom"", ""Hranice""]",0,"""Vánoční atmosféru dokresluje t…",null,5,"""ZPRÁVY""",[2],2,7,2020-11-29 00:00:00
"""https://www.idnes.cz/ostrava/z…","[""Ivana Lesková""]","""Fakulta v Ostravě nemůže přijí…","""Pro zájemce o studium medicíny…","[""Moravskoslezský kraj""]",2,"""Rozhodli o tom dnes členové Ra…",31,2,"""Kraje""",[2],2,4,2020-07-09 00:00:00


In [10]:
df03_test = pl.from_arrow(ds03["test"].data.table)
df03_test.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://teplicky.denik.cz/zloc…","[""Petr Málek""]","""Pytlačili na Vápence v Krupce,…","""Hlídka strážníků v Krupce lapi…",[],8,"""""Na operačním středisku byla p…",null,5,"""Krimi""",[1],1,5,2022-08-05 00:00:00
"""https://www.idnes.cz/kultura/d…","[""Tomáš Šťástka""]","""Dosud bavili na homeofficu, ny…","""Kontakt s diváky udržovali her…","[""Facebook"", ""Martin finger"", … ""Michal kern""]",4,"""Divadlo Letí chystá zmíněný au…",0,2,"""Kultura""",[1],1,7,2021-05-02 00:00:00
"""https://liberecky.denik.cz/cte…",[],"""Originální rapper Vojtaano pob…","""Na velkou dvoudenní oslavu stu…",[],0,"""Sobota 28. května je rockovějš…",null,5,null,[],0,5,2022-05-27 00:00:00
"""https://www.denik.cz/ze_sveta/…","[""Magdaléna Škapová""]","""Děti od krve, chaos a strach. …","""Střelba v malém texaském měste…","[""Střelba"", ""Děti"", … ""Masakr""]",1,"""„Viděli jsme malou holčičku, b…",null,5,"""Svět""",[2],2,3,2022-05-25 00:00:00
"""https://znojemsky.denik.cz/zpr…","[""Dagmar Sedláčková""]","""Kde seš ty atrapo, vítali se z…","""/VIDEO/ Na parkoviště u mostu …",[],0,"""Po skončení práce obdrží muži …",null,5,"""Znojemsko""",[2],2,4,2022-01-06 00:00:00


In [11]:
df04_train = pl.from_arrow(ds04["train"].data.table)
df04_train.sample(5)

sentence,choices,answer_idx
str,list[str],i64
"""Pak by si snad, či s mojí pomo…","[""řekla"", ""řeklo"", … ""řekl""]",2
"""Podklad nepochopitelně ____ op…","[""zůstávala"", ""zůstávalo"", … ""zůstával""]",4
"""Nějakých šedesát let ____ z vr…","[""kouzlila"", ""kouzlilo"", … ""kouzlil""]",2
"""Když jsem ____ pozvání do Bart…","[""dostala"", ""dostalo"", … ""dostal""]",0
"""A já jsem jenom ____ vždycky n…","[""odpovídala"", ""odpovídalo"", … ""odpovídal""]",4


In [12]:
df04_test = pl.from_arrow(ds04["test"].data.table)
df04_test.sample(5)

sentence,choices,answer_idx
str,list[str],i64
"""Cestou se ještě jejich jízdní …","[""spojila"", ""spojilo"", … ""spojil""]",3
"""Snažím se to kombinovat abych …","[""procvičila"", ""procvičilo"", … ""procvičil""]",0
"""V roce 2004 ____ podhůří Vysok…","[""postihla"", ""postihlo"", … ""postihl""]",0
"""Karajda 2004 V roce 2004 se po…","[""vrátila"", ""vrátilo"", … ""vrátil""]",4
"""Původní jméno Port Darwin ____…","[""byla"", ""bylo"", … ""byl""]",1


In [13]:
df05_train = pl.from_arrow(ds05["train"].data.table)
df05_train.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date,__index_level_0__,__index_level_1__
str,list[str],str,str,list[str],i64,str,f64,i64,str,list[i64],i64,i64,datetime[μs],i64,i64
"""https://www.irozhlas.cz/zpravy…",[],"""Ministerstvo poslalo ředitelům…","""Ministerstvo školství poslalo …","[""Koronavirus"", ""Koronavirus česko"", … ""Učitelé""]",2,"""Výběr pracovníků s přednostním…",null,6,"""Zprávy z domova""",[],0,3,2021-02-24 00:00:00,2,180
"""https://www.idnes.cz/ekonomika…","[""Veronika Bělohlávková""]","""Jeden kapr o Vánocích nestačí.…","""Češi si důležitost ryb a jejic…","[""Evropská unie"", ""Makro"", … ""Chko třeboňsko""]",5,"""Rybníkářství a chov kaprů má v…",46.0,2,"""Ekonomika""",[2],2,1,2021-12-06 00:00:00,7,188
"""https://www.idnes.cz/fotbal/re…","[""Miloslav Novák""]","""Německo se ztrapnilo až na kos…","""Jak ostudné. Jak trapné. Jak h…","[""Severní makedonie"", ""Uli hoeness"", … ""Timo werner""]",3,"""Německo v kvalifikaci o postup…",53.0,2,"""Fotbal""",[1],1,4,2021-04-01 00:00:00,3,29
"""https://www.irozhlas.cz/zpravy…",[],"""Vojáci USA při cvičení v Bulha…","""Bulharský majitel továrničky n…","[""Usa"", ""Bulharsko"", … ""Výsadkář""]",1,"""Podle agentury řídila americká…",null,6,"""Zprávy ze světa""",[],0,4,2021-06-03 00:00:00,1,11
"""https://www.seznamzpravy.cz/cl…",[],"""Statisíce lidí bez domova a tr…","""Víkendové devastující zemětřes…","[""Haiti"", ""Zemětřesení"", … ""Tropická bouře""]",1,"""„Země je fyzicky i mentálně zn…",null,1,"""Svět""",[],0,4,2021-08-19 00:00:00,1,126


In [14]:
df05_test = pl.from_arrow(ds05["test"].data.table)
df05_test.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date,__index_level_0__,__index_level_1__
str,list[str],str,str,list[str],i64,str,f64,i64,str,list[i64],i64,i64,datetime[μs],i64,i64
"""https://www.idnes.cz/ekonomika…",[],"""Ukrajina začala vyvážet elektř…","""Ukrajina začala vyvážet elektr…","[""Maďarsko"", ""Moldavsko"", … ""Kyjev""]",5,"""Zelenského vyjádření v jeho pr…",191.0,2,"""Ekonomika""",[],0,5,2022-07-01 00:00:00,7,94
"""https://www.seznamzpravy.cz/cl…","[""Adam Ryšánek""]","""Bla bla bla. Lídři nic nedělaj…","""Švédská aktivistka Greta Thunb…","[""Klima"", ""Glasgow"", … ""Klimatické změny""]",1,"""Ulicemi Glasgow v pátek pochod…",null,1,"""Svět""",[1],1,5,2021-11-05 00:00:00,1,108
"""https://www.idnes.cz/fotbal/po…","[""Miloslav Novák""]","""Žhavá odveta v Glasgow. Spartě…","""Jeden z těch velkých a atrakti…","[""Ac sparta praha"", ""Sk slavia praha"", … ""Glasgow""]",3,"""Důležité, zásadní, významné, r…",67.0,2,"""Fotbal""",[1],1,3,2021-11-24 00:00:00,3,65
"""https://www.seznamzpravy.cz/cl…",[],"""Na Kokořínsku začalo natáčení …","""V letošním srpnu a září režisé…","[""Film"", ""Praha"", … ""Koronavirus""]",2,"""Autorský snímek vypráví o Jarm…",null,1,"""Regiony""",[],0,4,2021-08-19 00:00:00,2,119
"""https://www.seznamzpravy.cz/cl…","[""Natálie Sousa""]","""Místo afghánských uprchlíků př…","""V sobotu večer vzlétlo z kábul…",[],1,"""Kolem 140 psů a 60 koček doraz…",null,1,"""Svět""",[2],2,7,2021-08-29 00:00:00,1,60


In [15]:
df06 = pl.from_arrow(ds06["test"].data.table)
df06.sample(5)

url,headline,brief,keywords,category,content,category_unclean
str,str,str,list[str],i64,str,str
"""https://www.idnes.cz/kultura/h…","""S Janem Vodňanským se naposled…","""Rodina, kolegové a přátelé se …","[""Pavel bém"", ""Jan hřebejk"", … ""Petr skoumal""]",4,"""Poslední rozloučení s Janem Vo…","""Kultura"""
"""https://magazin.aktualne.cz/ku…","""Don Juan jako fantasmagorie kr…","""Plameny žhnoucí touhy, nedosti…","[""Opera"", ""Don juan"", … ""Josif stalin""]",4,"""Plameny představují zcela jino…","""Klasická hudba"""
"""https://www.idnes.cz/ekonomika…","""Home Credit snížil počet zaměs…","""Skupina Home Credit musela po …","[""Čína"", ""Aljaška"", … ""Ks-jobdnes""]",5,"""Home Credit na svém webu uvádí…","""Ekonomika"""
"""https://svitavsky.denik.cz/kul…","""Janek Ledecký odstartoval v Li…","""/FOTO, VIDEO/ Janek Ledecký za…",[],4,"""„Je to normální průběh nemoci,…","""Kultura"""
"""https://www.novinky.cz/zahrani…","""Nová protilátka zcela blokuje …","""Protilátku, která brání replik…",[],1,"""O objevu informoval tým v časo…","""Zahraniční"""


### ***Creating a summary data file***

In [16]:
def build_unified_contexts(ds01, ds02, ds03, ds04, ds05, ds06):
    print("I am starting the unification of Czech text contexts using Polars...")
    
    # --- 1. News Corpus (Hynky v2 - 1.6M+ lines) ---
    # We glue the title, annotation and the entire body of the article into one massive block
    df_news = pl.from_arrow(ds03["train"].data.table).lazy().select([
        (pl.col("headline") + "\n" + pl.col("brief") + "\n" + pl.col("content")).alias("context"),
        pl.lit("news").alias("source")
    ])
    
    # --- 2. Legal / Alpaca dataset (czech-justice-summ-alpaca-long) ---
    # Here it makes sense to combine the instruction and the answer into one continuous text
    df_alpaca = pl.from_arrow(ds02["train"].data.table).lazy().select([
        (pl.col("instruction") + "\n" + pl.col("output")).alias("context"),
        pl.lit("alpaca_justice").alias("source")
    ])
    
    # --- 3. CERMAT Dataset (cermat_czech_mc) ---
    # Here we have the 'context' column, which we can enrich with a question
    df_cermat = pl.from_arrow(ds01["test"].data.table).lazy().select([
        (pl.col("context") + "\nOtázka: " + pl.col("query")).alias("context"),
        pl.lit("cermat").alias("source")
    ])

    # --- 4. Concatenation of all streams ---
    # Polars performs vertical joins extremely fast without unnecessary duplicate memory allocation
    unified_lazy = pl.concat([df_news, df_alpaca, df_cermat])
    
    # --- 5. Text extraction and cleaning + Calculation of logical predicates for JLNN ---
    final_df = unified_lazy.filter(
        pl.col("context").is_not_null() & (pl.col("context").str.len_chars() > 50)
    ).select([
        # Clean resulting text (context)
        pl.col("context"),
        pl.col("source"),
        
        # [PREDICT A]: Detection of masculine nouns in the text (Axiom of subject-predicate agreement)
        pl.col("context").str.to_lowercase().str.contains_any(
            ["chlapci", "muži", "dělníci", "studenti", "hráči", "učitelé", "lidi", "psi"]
        ).cast(pl.Float32).alias("has_masc_animate"),
        
        # [PREDICATE C]: Detection of subordinating conjunctions introducing subordinate clauses (Axiom of Punctuation)
        pl.col("context").str.to_lowercase().str.contains_any(
            ["že", "protože", "aby", "který", "jelikož", "jakmile", "zatímco"]
        ).cast(pl.Float32).alias("has_subord_conjunction"),
        
        # Auxiliary check: Is there a comma present in the text? (Basis for evaluating the t-norm)
        pl.col("context").str.contains(",").cast(pl.Float32).alias("has_comma")
    ]).collect()
    
    print(f"Unification complete! Total number of entire contexts in dataset: {final_df.height:,}")
    return final_df

In [17]:
%%time
df_dataset = build_unified_contexts(ds01, ds02, ds03, ds04, ds05, ds06)
df_dataset.sample(5)

I am starting the unification of Czech text contexts using Polars...
Unification complete! Total number of entire contexts in dataset: 1,646,599
CPU times: user 1min 17s, sys: 7.56 s, total: 1min 24s
Wall time: 1min 20s


context,source,has_masc_animate,has_subord_conjunction,has_comma
str,str,f32,f32,f32
"""Google pomalu privatizuje Andr…","""news""",1.0,1.0,1.0
"""Chávez se nechal kvůli chemote…","""news""",0.0,1.0,1.0
"""Kradená auta odváželi do kovoš…","""news""",1.0,1.0,1.0
"""Hráč od pánaboha i provokatér.…","""news""",1.0,1.0,1.0
"""Kontroverzní sKarty definitivn…","""news""",0.0,1.0,1.0


### ***Preparing the RAW dataset (Pure sentence extraction in Polars)***

In [18]:
def create_raw_sentence_dataset(unified_df):
    print("I extract clean sentence segments and classify their punctuation...")
    
    # This regex finds any characters that end with either a period, question mark, exclamation mark, comma,
    # or at the end of the entire text. This will keep the sign right at the end of the segment!
    regex_segment_with_punc = r"[^.\?!,]+[\.\?!,]?"

    raw_segments = (
        unified_df
        # 1. We extract all the matching sentence segments as a Word List
        .select(
            pl.col("context").str.extract_all(regex_segment_with_punc)
        )
        # 2. Expand the sheet into separate rows
        .explode("context")
        .rename({"context": "segment"})
        
        # 3. We clean up spaces at the beginning and end (the stuck punctuation will remain)
        .with_columns(pl.col("segment").str.strip_chars())
        
        # 4. FILTERING: We remove empty or too short things
        .filter(
            (pl.col("segment").str.len_chars() > 5) & 
            (pl.col("segment").str.len_chars() < 200) &
            pl.col("segment").str.contains(r"[a-zA-Zá-žÁ-Ž]")
        )
        
        # 5. CLASSIFICATION OF TERMINAL TYPE
        .with_columns([
            pl.when(pl.col("segment").str.ends_with("?")).then(pl.lit("otazník"))
            .when(pl.col("segment").str.ends_with("!")).then(pl.lit("vykřičník"))
            .when(pl.col("segment").str.ends_with(".")).then(pl.lit("tečka"))
            .when(pl.col("segment").str.ends_with(",")).then(pl.lit("čárka"))
            .otherwise(pl.lit("žádná"))
            .alias("punctuation_type")
        ])
        
        # 6. Random sample for our Ollama
        .sample(n=10000, with_replacement=False, seed=42)
    )
    
    print(f"Prepared {raw_segments.height} sentence segments for Ollama.")
    return raw_segments

In [19]:
%%time
raw_sentence_df = create_raw_sentence_dataset(df_dataset)
raw_sentence_df.head(5)

I extract clean sentence segments and classify their punctuation...
Prepared 10000 sentence segments for Ollama.
CPU times: user 1min 44s, sys: 6.85 s, total: 1min 51s
Wall time: 1min 50s


segment,punctuation_type
str,str
"""Pokud se to stává opakovaně,""","""čárka"""
"""Připomeňte si letošní změny: R…","""čárka"""
"""Ani to však značku patřící pod…","""tečka"""
"""dopadly do moře Severní Korea …","""čárka"""
"""„Někdy stačí i jen velká popel…","""čárka"""


### ***Export dataset***

In [20]:
def export_polars_to_pyarrow_dataset(polars_df, output_dir="processed_dataset"):
    print("Converting Polars DataFrame to PyArrow Table...")
    
    # We define a dictionary for translating Czech values ​​into English folder names
    translation_dict = {
        "otazník": "question_mark",
        "vykřičník": "exclamation_mark",
        "tečka": "period",
        "čárka": "comma",
        "žádná": "none"
    }
    
    # Temporarily add a column for folders in Polars and immediately convert to Arrow
    arrow_table = (
        polars_df
        .with_columns(
            pl.col("punctuation_type").replace(translation_dict).alias("folder_name")
        )
        .to_arrow()
    )
    
    print(f"PyArrow table ready. Number of rows: {arrow_table.num_rows:,}")
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Saving PyArrow Dataset to folder '{output_dir}' with English names...")
    
    # Writing to disk - partition will be done according to "folder_name" (English)
    ds.write_dataset(
        data=arrow_table,
        base_dir=output_dir,
        format="parquet",
        partitioning=["folder_name"],
        use_threads=True,
        existing_data_behavior="overwrite_or_ignore"
    )
    
    print("Export completed successfully!")

In [21]:
%%time
export_polars_to_pyarrow_dataset(raw_sentence_df, output_dir="raw_segments_dataset")

Converting Polars DataFrame to PyArrow Table...
PyArrow table ready. Number of rows: 10,000
Saving PyArrow Dataset to folder 'raw_segments_dataset' with English names...
Export completed successfully!
CPU times: user 29.1 ms, sys: 9.17 ms, total: 38.3 ms
Wall time: 60.8 ms
